# 02 — Data Cleaning & Standardization

Purpose:
Transform the raw Fragrantica dataset into a clean, standardized dataset ready for EDA and modeling.

This notebook:
- Cleans and standardizes column formats
- Converts rating to numeric
- Fixes string formatting (trim, consistent casing where needed)
- Handles missing values (Year, perfumers, accords)
- Resolves duplicates (Perfume + Brand)
- Saves a cleaned dataset to `data/processed/`

Outputs:
- `data/processed/fragrantica_clean.csv`

In [2]:
# Import necessary libraries

from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
# Define paths

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "fragrantica_raw.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_PATH = PROCESSED_DIR / "fragrantica_clean.csv"

RAW_PATH, CLEAN_PATH


(WindowsPath('c:/Users/swara/olfactory-intelligence/data/raw/fragrantica_raw.csv'),
 WindowsPath('c:/Users/swara/olfactory-intelligence/data/processed/fragrantica_clean.csv'))

In [ ]:
# Load raw data

df_raw = pd.read_csv(RAW_PATH, sep=";", encoding="ISO-8859-1")
df_raw.shape

# Make a copy of raw data
df = df_raw.copy()

(24063, 18)

In [ ]:
# Standardize column names

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_", regex=False)
)

df.columns

Index(['url', 'perfume', 'brand', 'country', 'gender', 'rating_value',
       'rating_count', 'year', 'top', 'middle', 'base', 'perfumer1',
       'perfumer2', 'mainaccord1', 'mainaccord2', 'mainaccord3', 'mainaccord4',
       'mainaccord5'],
      dtype='object')

In [ ]:
# Trim whitespace in all object columns 

obj_cols = df.select_dtypes(include="object").columns
df[obj_cols] = df[obj_cols].apply(lambda s: s.astype(str).str.strip().replace({"": pd.NA, "nan": pd.NA}))

In [ ]:
# Convert rating_value to numeric (comma → dot)

df["rating_value"] = (
    df["rating_value"]
      .astype(str)
      .str.replace(",", ".", regex=False)
)

df["rating_value"] = pd.to_numeric(df["rating_value"], errors="coerce")
df["rating_value"].describe()


nulls            0
zeros            0
negatives        0
min             26
max          29858
dtype: int64


In [8]:
# Ensure rating_count is numeric and valid

df["rating_count"] = pd.to_numeric(df["rating_count"], errors="coerce").astype("Int64")

pd.Series({
    "nulls": int(df["rating_count"].isna().sum()),
    "zeros": int((df["rating_count"] == 0).sum()),
    "negatives": int((df["rating_count"] < 0).sum()),
    "min": int(df["rating_count"].min()),
    "max": int(df["rating_count"].max()),
})

nulls            0
zeros            0
negatives        0
min             26
max          29858
dtype: int64

In [17]:
# Clean Year Column

df["year"] = pd.to_numeric(df["year"], errors="coerce")

# Optional sanity check for unrealistic years
df.loc[(df["year"] < 1700) | (df["year"] > 2026), "year"] = pd.NA

# Convert to nullable integer
df["year"] = df["year"].astype("Int64")

pd.Series({
    "nulls": int(df["year"].isna().sum()),
    "min": int(df["year"].min()) if df["year"].notna().any() else None,
    "max": int(df["year"].max()) if df["year"].notna().any() else None,
})

nulls    2015
min      1781
max      2024
dtype: int64

In [ ]:
# Standardize "unknown" in perfumers 

for col in ["perfumer1", "perfumer2"]:
    df[col] = (
        df[col]
          .astype(str)
          .str.strip()
          .str.lower()
          .replace({"unknown": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA, "": pd.NA})
    )

df[["perfumer1", "perfumer2"]].isna().mean().round(3)

perfumer1    0.512
perfumer2    0.944
dtype: float64

In [16]:
# Clean Gender Column

df["gender"] = (
    df["gender"]
        .astype(str)
        .str.strip()
        .str.lower()
)

# Keep only valid categories
valid_genders = {"men", "women", "unisex"}
df.loc[~df["gender"].isin(valid_genders), "gender"] = pd.NA

# Convert to categorical type
df["gender"] = df["gender"].astype("category")

df["gender"].value_counts(dropna=False)

gender
women     11286
unisex     7652
men        4908
Name: count, dtype: int64

In [ ]:
# Standardize country + brand + perfume text

df["brand"] = df["brand"].str.lower()
df["country"] = df["country"].str.strip()

In [13]:
# Handle Perfume+Brand duplicates

before = df.shape[0]

df = df.sort_values(["brand", "perfume", "rating_count"], ascending=[True, True, False])
df = df.drop_duplicates(subset=["brand", "perfume"], keep="first")

after = df.shape[0]

before, after, before - after



(24063, 23846, 217)

In [ ]:
# Reorder columns 

ordered_cols = [
    "url", "perfume", "brand", "country", "gender",
    "rating_value", "rating_count", "year",
    "top", "middle", "base",
    "perfumer1", "perfumer2",
    "mainaccord1", "mainaccord2", "mainaccord3", "mainaccord4", "mainaccord5"
]

df = df[ordered_cols]

In [ ]:
# Final missingness check

(df.isna().mean() * 100).sort_values(ascending=False).round(2).head(15)

perfumer2       94.46
perfumer1       51.14
year             8.45
mainaccord5      4.06
mainaccord4      1.60
mainaccord3      0.47
mainaccord2      0.05
base             0.00
mainaccord1      0.00
url              0.00
perfume          0.00
top              0.00
rating_count     0.00
rating_value     0.00
gender           0.00
dtype: float64

In [ ]:
# Final dtypes

df.dtypes

url              object
perfume          object
brand            object
country          object
gender           object
rating_value    float64
rating_count      Int64
year            float64
top              object
middle           object
base             object
perfumer1        object
perfumer2        object
mainaccord1      object
mainaccord2      object
mainaccord3      object
mainaccord4      object
mainaccord5      object
dtype: object

In [19]:
# Save cleaned dataset

df.to_csv(CLEAN_PATH, index=False, encoding="utf-8")
CLEAN_PATH

WindowsPath('c:/Users/swara/olfactory-intelligence/data/processed/fragrantica_clean.csv')

In [20]:
# Reload to confirm save

df_check = pd.read_csv(CLEAN_PATH)
df_check.shape, df_check.head(3)

((23846, 18),
                                                  url              perfume  \
 0  https://www.fragrantica.com/perfume/a-dozen-ro...          amber-queen   
 1  https://www.fragrantica.com/perfume/a-dozen-ro...           angel-face   
 2  https://www.fragrantica.com/perfume/a-dozen-ro...  shakespeare-in-love   
 
            brand country gender  rating_value  rating_count    year  \
 0  a-dozen-roses     USA  women          3.77            96  2012.0   
 1  a-dozen-roses     USA  women          3.96           107  2013.0   
 2  a-dozen-roses     USA  women          3.62           109  2011.0   
 
                            top                               middle  \
 0  apricot, clementine, ginger                    damask rose, rose   
 1         black currant, apple  rose, violet, peony, lilac, jasmine   
 2                         pear              rose, gardenia, jasmine   
 
                              base    perfumer1 perfumer2 mainaccord1  \
 0                 